In [ ]:
# 1. Uninstall everything first to remove conflicts
!pip uninstall -y torch torchvision torchaudio peft transformers accelerate bitsandbytes trl

# 2. Reinstall compatible versions (PyTorch 2.1+ is recommended for Llama 2)
# We install torch first to ensure the base is correct
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# 3. Install the specific fine-tuning libraries
!pip install -U peft transformers accelerate bitsandbytes trl datasets scipy

In [ ]:
from huggingface_hub import login

# Replace with your actual token starting with 'hf_...'
login(token="")

In [6]:
import torch
from trl import SFTTrainer

class EWCTrainer(SFTTrainer):
    def __init__(self, fisher_dict, opt_param_dict, ewc_lambda, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.fisher_dict = fisher_dict
        self.opt_param_dict = opt_param_dict
        self.ewc_lambda = ewc_lambda

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        # 1. Compute Standard Loss
        if num_items_in_batch is not None:
             outputs = model(**inputs)
        else:
             outputs = model(**inputs)
        
        loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]

        # --- THE FIX STARTS HERE ---
        # Initialize ewc_loss on the SAME device as the main loss
        ewc_loss = torch.tensor(0.0, device=loss.device)
        
        for name, param in model.named_parameters():
            if name in self.fisher_dict:
                # 1. Move saved tensors to the current parameter's device for calculation
                opt_param = self.opt_param_dict[name].to(param.device)
                fisher_val = self.fisher_dict[name].to(param.device)
                
                # 2. Calculate the penalty term (Result is on param.device)
                term = (fisher_val * (param - opt_param) ** 2).sum()
                
                # 3. Move the result to the main loss device BEFORE adding
                ewc_loss += term.to(loss.device)
        # --- THE FIX ENDS HERE ---

        total_loss = loss + (self.ewc_lambda / 2) * ewc_loss
        
        return (total_loss, outputs) if return_outputs else total_loss

In [7]:
import torch
import gc
import os
import shutil
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
from peft import PeftModel, prepare_model_for_kbit_training
from trl import SFTConfig
from datasets import load_dataset

# --- 1. CONFIGURATION & FILE SETUP ---
# Define the writable directory for the Task 1 Adapters
TASK1_FOLDER = "/kaggle/working/llama2-hp-task1-adapter"
os.makedirs(TASK1_FOLDER, exist_ok=True)

# Copy uploaded files to the writable folder
# (Update these filenames if you named them differently in your upload)
files_to_copy = ["adapter_model.safetensors", "adapter_config.json"]
input_dir = "." # Assuming files are in current directory, change to /kaggle/input/... if needed

for filename in files_to_copy:
    src = os.path.join(input_dir, filename)
    dst = os.path.join(TASK1_FOLDER, filename)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"✅ Copied {filename} -> {dst}")
    elif os.path.exists(os.path.join("/kaggle/input", filename)): # Check generic input
         shutil.copy2(os.path.join("/kaggle/input", filename), dst)
         print(f"✅ Copied {filename} from input -> {dst}")
    else:
        # If files are already there from previous steps, we are good
        if os.path.exists(dst):
            print(f"✅ Found {filename} in target folder.")
        else:
            print(f"⚠️ WARNING: Could not find {filename}. Model loading might fail.")

# File Paths
DATASET_FILE = "/kaggle/input/continual-learningtask1/harry_potter_train2_instruct.json"       # <--- Ensure this is uploaded
FISHER_FILE = "/kaggle/input/continual-learningtask1/fisher_task1.pt"                # <--- Ensure this is uploaded
BASE_MODEL_ID = "microsoft/Llama2-7b-WhoIsHarryPotter"
NEW_ADAPTER_NAME = "llama2-hp-task2-adapter"

# --- 2. MEMORY CLEANUP ---
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
torch.cuda.empty_cache()
gc.collect()

# --- 3. LOAD DATASET ---
# If this fails, ensure the json file is uploaded
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")

def format_instruction(sample):
    return f"<s>[INST] {sample['instruction']} [/INST] {sample['output']} </s>"

# --- 4. LOAD BASE MODEL (4-bit) ---
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, 
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16 
)

# --- 5. LOAD TASK 1 ADAPTERS (CONTINUAL LEARNING) ---
print(f"Loading Task 1 weights from '{TASK1_FOLDER}'...")
# is_trainable=True is CRITICAL. It tells PEFT to continue updating these weights.
model = PeftModel.from_pretrained(base_model, TASK1_FOLDER, is_trainable=True)

# Prepare model for training (Casts norms to fp32)
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# --- 6. LOAD EWC CONSTRAINTS ---
print(f"Loading Fisher Matrix from '{FISHER_FILE}'...")
checkpoint = torch.load(FISHER_FILE)
fisher_matrix = checkpoint['fisher']
opt_params = checkpoint['opt_params']
print("✅ EWC Constraints loaded.")

# --- 7. TRAINING CONFIGURATION ---
sft_config = SFTConfig(
    output_dir="./results_task2",
    dataset_text_field="output",
    max_length=512,  
    packing=False,
    
    num_train_epochs=3,
    per_device_train_batch_size=1, 
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    
    # SAFETY SETTINGS (Prevent T4 Crashes)
    fp16=False, 
    bf16=False,
    
    learning_rate=2e-4,
    weight_decay=0.001,
    logging_steps=25,
    save_strategy="epoch",
    report_to="none"
)

# --- 8. INITIALIZE CUSTOM EWC TRAINER ---
# EWC_LAMBDA determines how hard the model tries to remember Book 1.
# 0.4 is a balanced starting point.
EWC_LAMBDA = 0.4

trainer = EWCTrainer(
    model=model,
    train_dataset=dataset,
    fisher_dict=fisher_matrix,
    opt_param_dict=opt_params,
    ewc_lambda=EWC_LAMBDA,
    formatting_func=format_instruction,
    processing_class=tokenizer,
    args=sft_config,
)

# --- 9. TRAIN & SAVE ---
print("🚀 Starting Task 2 Training (Chamber of Secrets)...")
trainer.train()

# Save the new adapters (Task 2)
trainer.model.save_pretrained(NEW_ADAPTER_NAME)
tokenizer.save_pretrained(NEW_ADAPTER_NAME)
print(f"✅ Task 2 Complete. New adapters saved to '{NEW_ADAPTER_NAME}'")

✅ Found adapter_model.safetensors in target folder.
✅ Found adapter_config.json in target folder.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading Task 1 weights from '/kaggle/working/llama2-hp-task1-adapter'...
Loading Fisher Matrix from '/kaggle/input/continual-learningtask1/fisher_task1.pt'...
✅ EWC Constraints loaded.


Tokenizing train dataset:   0%|          | 0/174 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/174 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


🚀 Starting Task 2 Training (Chamber of Secrets)...


Step,Training Loss
25,8.269550
50,8.209634
75,8.347968
100,8.123022
125,8.300010


✅ Task 2 Complete. New adapters saved to 'llama2-hp-task2-adapter'


In [11]:
import torch
import json
import pandas as pd
from tqdm import tqdm

# --- Configuration ---
QA_FILE = "/kaggle/input/qa-testing/Harry_porter_book1_qa_pairs.txt"
OUTPUT_CSV = "task1_eval_concise.csv"

print(f"Evaluating {QA_FILE} with 'Gist' constraints...")

# 1. Load Data
with open(QA_FILE, "r") as f:
    try:
        qa_data = json.load(f)
    except json.JSONDecodeError:
        print("❌ Error reading JSON. Please check the file.")
        qa_data = []

# 2. Evaluation Loop
results = []
trainer.model.eval() # Ensure model is in eval mode

for entry in tqdm(qa_data):
    question = entry["question"]
    expected_answer = entry["answer"]
    
    # --- PROMPT ENGINEERING FOR CONCISENESS ---
    # We add instructions inside the [INST] tag to force brevity.
    prompt = f"<s>[INST] Answer the following question concisely in one or two sentences. Do not continue the story.\n\nQuestion: {question} [/INST]"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(trainer.model.device)
    
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=60,    # STRICT LIMIT: Prevents long rambling
            do_sample=True,
            temperature=0.1,      # LOW TEMP: Makes model factual and focused (less "storytelling")
            top_p=0.9,
            repetition_penalty=1.2, # Penalize repeating itself
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Decode and Clean
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    if "[/INST]" in full_text:
        model_answer = full_text.split("[/INST]")[-1].strip()
    else:
        model_answer = full_text
        
    results.append([question, expected_answer, model_answer])

# 3. Save to CSV
df = pd.DataFrame(results, columns=["Question", "Expected Answer", "Model Answer"])
df.to_csv(OUTPUT_CSV, index=False)

print(f"\n✅ Evaluation Complete! Saved to {OUTPUT_CSV}")
print("\n--- Sample Results ---")
print(df.head())

Evaluating /kaggle/input/qa-testing/Harry_porter_book1_qa_pairs.txt with 'Gist' constraints...


100%|██████████| 20/20 [01:18<00:00,  3.91s/it]


✅ Evaluation Complete! Saved to task1_eval_concise.csv

--- Sample Results ---
                                            Question  \
0                        Where do the Dursleys live?   
1         What company does Vernon Dursley work for?   
2  Why do the Dursleys fear being associated with...   
3  What unusual behavior of animals is noticed on...   
4  What strange sight does Vernon Dursley see on ...   

                                     Expected Answer  \
0  They live at number four, Privet Drive, in Lit...   
1  He is the director of Grunnings, a company tha...   
2  They believe the Potters are strange and invol...   
3           Owls are seen flying during the daytime.   
4    He sees a cat that appears to be reading a map.   

                                        Model Answer  
0  The Dursleys lived at number four, Privet Driv...  
1  Sure thing! Here is my answer to your question...  
2  The Dursleys feared being associated with the ...  
3  On the day the story be